In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
!cp -r "/content/drive/MyDrive/rock_dataset_split" /content/
!ls /content/rock_dataset_split

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data.yaml  test  train	valid


In [ ]:
root = "/content/rock_dataset_split"
open(f"{root}/data.yaml","w").write(
f"""path: {root}
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['ROCK']
""")
print(open(f"{root}/data.yaml").read())

path: /content/rock_dataset_split
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['ROCK']



In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
results = model.train(
    data="/content/rock_dataset_split/data.yaml",
    epochs=300, patience=50, imgsz=640, batch=16,

    degrees=20,
    fliplr=0.5, flipud=0.5,
    scale=0.5,
    translate=0.1,
    shear=3.0,
    perspective=0.0005,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.4,
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.15,
    copy_paste=0.1,
    erasing=0.4,


    cos_lr=True, weight_decay=0.0005, dropout=0.1,
    optimizer="auto", amp=True, cache=True, seed=0, plots=True,
)
print("result:", results.save_dir)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.106 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/rock_dataset_split/data.yaml, degrees=20, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=

In [ ]:
m = model.val(split="test")
print("mAP50 =", round(m.box.map50,4), " mAP50-95 =", round(m.box.map,4),
      " P =", round(m.box.mp,4), " R =", round(m.box.mr,4))

Ultralytics 8.4.106 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 66.3±42.9 MB/s, size: 372.5 KB)
val: Scanning /content/rock_dataset_split/test/labels... 78 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 78/78 222.7it/s 0.4s
val: New cache created: /content/rock_dataset_split/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.2it/s 4.1s
                   all         78         88      0.852      0.785      0.897      0.479
Speed: 8.7ms preprocess, 11.2ms inference, 0.0ms loss, 3.6ms postprocess per image
Results saved to /content/runs/detect/val
mAP50 = 0.8967  mAP50-95 = 0.4795  P = 0.8519  R = 0.7846


In [ ]:
import glob, os, shutil
best = max(glob.glob("/content/**/weights/best.pt", recursive=True), key=os.path.getmtime)
shutil.copy(best, "/content/drive/MyDrive/rock_yolo11_best.pt")
shutil.make_archive("/content/drive/MyDrive/rock_yolo11_run", "zip", results.save_dir)
print("save as:", best)

save as: /content/runs/detect/train/weights/best.pt


In [ ]:
import os, glob

root = "/content/rock_dataset_split"
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

total_img = total_box = 0
print(f"{'split':6} {'images':>8} {'boxes':>8} {'boxes/img':>10}")
print("-" * 36)
for split in ("train", "valid", "test"):
    img_dir = os.path.join(root, split, "images")
    lbl_dir = os.path.join(root, split, "labels")

    imgs = [f for f in glob.glob(os.path.join(img_dir, "*")) if f.lower().endswith(IMG_EXT)]
    n_img = len(imgs)

    n_box = 0
    for txt in glob.glob(os.path.join(lbl_dir, "*.txt")):
        with open(txt) as f:
            n_box += sum(1 for line in f if line.strip())

    total_img += n_img
    total_box += n_box
    ratio = n_box / n_img if n_img else 0
    print(f"{split:6} {n_img:>8} {n_box:>8} {ratio:>10.2f}")

print("-" * 36)
ratio = total_box / total_img if total_img else 0
print(f"{'TOTAL':6} {total_img:>8} {total_box:>8} {ratio:>10.2f}")